# First test of the generator-aware checker

Does a small checker get better when it also reads the answering model's own signals: how sure
the model was of each word, and whether it was looking at the passages when it wrote it?

1. **Record the signals (GPU).** RAGTruth's test answers written by open models (Llama-2-7B-chat,
   Mistral-7B-Instruct, Llama-2-13B-chat; 450 answers each) are re-read once by the model that
   wrote them, loaded in 4-bit. For every answer token we keep 4 confidence numbers and the
   lookback ratio of every layer. Small files go to `MyDrive/ondevice_hallu/work/signals`.
2. **Combine and score (CPU).** For TinyLettuce 17M/32M/68M and LettuceDetect-base (their test
   predictions from the size ladder), a small model learns to combine the text score with the
   signals. Every version (text only, + confidence, + attention, + both) is trained and scored the
   same way: 5-fold cross-validation, folds split by source. Span F1 at the 0.5 cut-off.

| Step | Needs | Time on a free T4 session (estimate) |
|---|---|---|
| Setup | internet | 3 min |
| 1. Signals, 3 answering models | GPU | 45-75 min (downloads included) |
| 2. Combine and score | CPU | about 5 min |

Select **Runtime > Change runtime type > T4 GPU** and run the cells in order. Finished parts are
skipped, so after a disconnect run the setup cells again and then the step that was running.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, sys, json, subprocess, shutil, glob, zipfile
from pathlib import Path
from IPython.display import Markdown, display

REPO = 'https://github.com/gbirhanu/ondevice-hallu.git'
CODE = Path('/content/ondevice_hallu')
DRIVE = Path('/content/drive/MyDrive/ondevice_hallu')
WORK = DRIVE / 'work'                       # the size-ladder results live here
DATA = Path('/content/RAGTruth')

def sh(command):
    result = subprocess.run(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                            env={**os.environ, 'GIT_TERMINAL_PROMPT': '0'})
    print(result.stdout[-3000:], flush=True)
    return result.returncode

if (CODE / '.git').exists():
    sh(['git', '-C', str(CODE), 'pull', '--ff-only'])
else:
    assert sh(['git', 'clone', REPO, str(CODE)]) == 0, f'Cannot clone {REPO}: push the code first.'
if not DATA.exists():
    sh(['git', 'clone', '--depth', '1', 'https://github.com/ParticleMedia/RAGTruth.git', str(DATA)])
sh([sys.executable, '-m', 'pip', 'install', '-q', 'transformers==4.57.6', 'bitsandbytes', 'accelerate',
    'scikit-learn'])
missing = [n for n in ('tiny17m', 'tiny32m', 'tiny68m', 'ld_base') if not (WORK / 'preds' / f'{n}_test.json.gz').exists()]
print('Text-checker predictions missing for:', missing or 'none (good)')

In [ ]:
def run(*arguments):
    """Runs one run.py command and shows its output as it comes. Returns True when it worked."""
    command = [sys.executable, '-u', str(CODE / 'run.py'), *map(str, arguments),
               '--data-dir', str(DATA), '--work-dir', str(WORK)]
    process = subprocess.Popen(command, cwd=CODE, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                               env={**os.environ, 'TOKENIZERS_PARALLELISM': 'false'})
    for line in process.stdout:
        if 'Loading checkpoint shards' not in line and 'Warning' not in line:
            print(line, end='', flush=True)
    return process.wait() == 0

def free_disk():
    """Delete downloaded model weights (each answering model is 13-26 GB)."""
    hub = Path(os.environ.get('HF_HOME', Path.home() / '.cache' / 'huggingface')) / 'hub'
    for d in hub.glob('models--*'):
        shutil.rmtree(d, ignore_errors=True)

## Optional: a Hugging Face token for Mistral

The two Llama 2 models download without sign-in. Mistral-7B-Instruct needs a free Hugging Face
account that has accepted its licence on <https://huggingface.co/mistralai/Mistral-7B-Instruct-v0.1>,
and a token from <https://huggingface.co/settings/tokens> saved in Colab's **Secrets** panel (the key
icon on the left) as `HF_TOKEN`, with notebook access switched on. Without it, Mistral is skipped
and the test runs on the two Llama models (900 answers), which is still enough.

In [ ]:
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    print('Hugging Face token found.')
except Exception:
    print('No Hugging Face token: Mistral will be skipped unless it is already recorded.')

## Step 1: record the answering models' signals (GPU)

In [ ]:
import torch
assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > T4 GPU for this step.'
for generator in ['llama-2-7b-chat', 'mistral-7B-instruct', 'llama-2-13b-chat']:
    ok = run('signals', '--generator', generator)
    if not ok:
        print(f'--> {generator} did not finish; the next step uses whatever was recorded.')
    free_disk()

## Step 2: does reading the answering model help? (CPU)

Paste the table back into the chat.

In [ ]:
assert run('fusion'), 'The fusion step failed; see the message above.'
display(Markdown((WORK / 'results' / 'fusion.md').read_text()))

**How to read it.** Compare each checker's `text` row (its own score, re-learned the same way) with
`text+confidence` and `text+confidence+attention`. The last column is the gain and its 95%
interval. A gain of about 2 points or more, with the interval above zero, is the green light for
building the full checker. The `(none)` rows use the signals without any text checker.